In [1]:
!nvidia-smi

Thu Oct  8 07:06:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from numba import cuda

device = cuda.get_current_device()
name = device.name

print(f"Device Name: {name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


In [6]:
import numpy as np
import time
from numba import cuda

def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [7]:
# warmup compile once
tmp = cuda.to_device(np.ones(10, dtype=np.float32))
out = cuda.device_array_like(tmp)
double_gpu_kernel[1, 32](tmp, out)
cuda.synchronize()

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [8]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # CPU time (skip huge)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # GPU total (H2D + kernel + D2H)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()   # includes sync for copy, but kernel is async -> still safe with next line
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU compute-only (data already on GPU)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [9]:
SIZES = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

# лучше: seed = твой StudentID как число (если он числовой)
STUDENT_SEED = 1234  # <-- замени на свой ID (или его часть), чтобы было "твое"
rows = []
for N in SIZES:
    cpu_ms, gpu_total_ms, gpu_comp_ms = benchmark_run(N, STUDENT_SEED)
    winner = "GPU" if (not np.isnan(cpu_ms) and gpu_total_ms < cpu_ms) else "CPU"
    rows.append((N, cpu_ms, gpu_total_ms, gpu_comp_ms, winner))

for r in rows:
    N, cpu_ms, gt, gc, w = r
    print(f"N={N:<10} CPU={cpu_ms:>10.3f} ms | GPU_total={gt:>10.3f} ms | GPU_kernel={gc:>10.3f} ms | {w}")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


N=100        CPU=     0.093 ms | GPU_total=     1.730 ms | GPU_kernel=     0.112 ms | CPU
N=1000       CPU=     2.847 ms | GPU_total=     5.845 ms | GPU_kernel=     0.134 ms | CPU
N=10000      CPU=    11.773 ms | GPU_total=     5.025 ms | GPU_kernel=     0.110 ms | GPU
N=100000     CPU=   229.656 ms | GPU_total=     4.370 ms | GPU_kernel=     0.124 ms | GPU
N=1000000    CPU=   758.799 ms | GPU_total=     5.951 ms | GPU_kernel=     0.157 ms | GPU
N=5000000    CPU=       nan ms | GPU_total=    15.504 ms | GPU_kernel=     0.308 ms | CPU
N=10000000   CPU=       nan ms | GPU_total=    27.752 ms | GPU_kernel=     0.493 ms | CPU


In [11]:
import numpy as np
from numba import cuda

@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    d_arr[idx] = 999.0  # no bounds check -> illegal access

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4  # 1024 threads total

print("Launching defective kernel ...")
try:
    defective_kernel[blocks_per_grid, threads_per_block](d_arr)
    cuda.synchronize()
    print("Kernel finished. Copying back...")
    h_result = d_arr.copy_to_host()
    print("Last element:", h_result[-1])
except Exception as e:
    print("Observed error/behavior:", repr(e))

Launching defective kernel ...
Kernel finished. Copying back...
Last element: 999.0


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [12]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block  # 5
total_threads = blocks_per_grid * threads_per_block                  # 40

log_b = cuda.device_array(total_threads, dtype=np.int32)
log_t = cuda.device_array(total_threads, dtype=np.int32)
log_g = cuda.device_array(total_threads, dtype=np.int32)
log_a = cuda.device_array(total_threads, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

hb = log_b.copy_to_host()
ht = log_t.copy_to_host()
hg = log_g.copy_to_host()
ha = log_a.copy_to_host()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<14}")
print("-" * 56)
for i in range(32, 40):
    status = "ACTIVE" if ha[i] == 1 else "IDLE (GUARDED)"
    print(f"{hg[i]:<10} | {hb[i]:<10} | {ht[i]:<10} | {status:<14}")

Global ID  | Block ID   | Thread ID  | Status        
--------------------------------------------------------
32         | 4          | 0          | ACTIVE        
33         | 4          | 1          | ACTIVE        
34         | 4          | 2          | ACTIVE        
35         | 4          | 3          | ACTIVE        
36         | 4          | 4          | ACTIVE        
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [14]:
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]   # race

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N = 50_000
h_arr = np.ones(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads = 256
blocks = (N + threads - 1) // threads

def run_kernel(kernel, trials=5):
    outs = []
    times = []
    for _ in range(trials):
        d_total = cuda.to_device(np.array([0.0], dtype=np.float32))
        t0 = time.perf_counter()
        kernel[blocks, threads](d_arr, d_total)
        cuda.synchronize()
        ms = (time.perf_counter() - t0) * 1000.0
        outs.append(d_total.copy_to_host()[0])
        times.append(ms)
    return outs, times

naive_outs, naive_times = run_kernel(naive_sum_kernel, trials=5)
print("Naive outputs:", naive_outs)
print("Naive times (ms):", naive_times)

atomic_outs, atomic_times = run_kernel(atomic_sum_kernel, trials=1)
print("Atomic output:", atomic_outs[0])
print("Atomic time (ms):", atomic_times[0])

Naive outputs: [np.float32(2.0), np.float32(2.0), np.float32(2.0), np.float32(2.0), np.float32(2.0)]
Naive times (ms): [58.3767150001222, 0.1580350003678177, 0.11011500009772135, 0.10159500016015954, 0.10694700040403404]
Atomic output: 50000.0
Atomic time (ms): 45.96138399983829


In [16]:
import math

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx*dx + dy*dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)
blocks_2d = ((WIDTH + threads_2d[0] - 1) // threads_2d[0],
            (HEIGHT + threads_2d[1] - 1) // threads_2d[1])

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH//2, HEIGHT//2)
cuda.synchronize()

result = d_canvas.copy_to_host()
print("blocks_2d:", blocks_2d, " total blocks:", blocks_2d[0]*blocks_2d[1])
print("Center (512,512):", float(result[512, 512]))
print("Corner (0,0):", float(result[0, 0]))

blocks_2d: (64, 64)  total blocks: 4096
Center (512,512): 1.0
Corner (0,0): 0.0


In [20]:
import hashlib
from numba import cuda

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))

    dev_name = cuda.get_current_device().name
    if isinstance(dev_name, str):
        dev_name = dev_name.encode('utf-8')
    h.update(dev_name)

    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))

    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest


STUDENT_ID = "230103374"
CROSSOVER_N = 10000
token = generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 3CE433FC0272B1DD
